# Pediatrik Posterior Fossa Tümörü — Sınıflandırıcı Eğitimi (Colab / GPU)

Kod GitHub'dan (`iakdulumgazi/pf-tumor`) tek komutla çekilir; `src`'yi elle yüklemene gerek yok.
Preprocessing / BraTS-PEDs segmentasyonu / ROI kırpma **yerelde** yapılır; Colab yalnızca
**sınıflandırıcı eğitimini** üstlenir.

## İki mod

1. **DEMO MODU** — sentetik ROI üretir, gerçek veri/Drive GEREKMEZ. Aç → GPU seç → sırayla çalıştır.
   Eğitim döngüsünün GPU'da çalıştığını doğrulamak için.
2. **GERÇEK MOD** — kendi ROI'lerini (`data/roi/*.npy` + `labels.csv`) Drive'dan okur.

## ⚠️ Veri gizliliği

- **CBTN verisini Colab'a/Drive'a YÜKLEME** (DUA Madde 6 — controlled-access). CBTN eğitimi
  **yerel GPU'da** (ör. RTX 5070); aynı kod orada da çalışır.
- Colab yalnızca **demo** ya da **kurumsal/etik onaylı kendi anonim verin** için.
- Kod public, ama repoya hasta verisi/model asla girmez (`.gitignore`).

## 1. Ortam ve kod

In [ ]:
import torch
ok = torch.cuda.is_available()
print('GPU var mı:', ok)
print('Cihaz:', torch.cuda.get_device_name(0) if ok else 'YOK — Runtime > Change runtime type > T4 GPU seç, sonra bu hücreyi tekrar çalıştır!')

In [ ]:
# Kodu GitHub'dan çek (public) + MONAI kur
![ -d /content/pf-tumor ] || git clone -q https://github.com/iakdulumgazi/pf-tumor.git /content/pf-tumor
!pip install -q monai

import sys
sys.path.append('/content/pf-tumor')
from src.classification.train import run
from src.classification.model import build_model, count_params, CLASSES
print('Kod yüklendi. Sınıflar:', CLASSES, '| model parametreleri:', f'{count_params(build_model()):,}')

## 2A. DEMO MODU — sentetik veriyle eğitimi test et

Gerçek veri/Drive gerekmez. Her sınıfa (MB/PA/EP) ayırt edici sinyal gömülür; model bunu
öğrenebilmeli (macro-F1 yükselmeli). **Gerçek verin varsa bu iki hücreyi atla, 2B'ye geç.**

In [ ]:
import numpy as np, csv
from pathlib import Path

DEMO = Path('/content/demo'); (DEMO / 'roi').mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(0)
patterns = {'MB': 2, 'PA': 0, 'EP': 3}   # her sınıf farklı kanalda parlak
rows, pid = [], 0
for cls, ch in patterns.items():
    for _ in range(8):                    # sınıf başına 8 hasta
        pid += 1
        vol = (rng.random((4, 64, 64, 64)) * 0.5).astype('float32')
        c = 32; vol[ch, c-8:c+8, c-8:c+8, c-8:c+8] += 2.0
        case = f'P{pid:03d}'
        np.save(DEMO / 'roi' / f'{case}_roi.npy', vol)
        rows.append({'case_id': case, 'patient_id': case, 'label': cls})
with open(DEMO / 'labels.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['case_id', 'patient_id', 'label']); w.writeheader(); w.writerows(rows)
print(f'{len(rows)} sentetik ROI üretildi -> {DEMO}')

In [ ]:
summary = run(
    roi_dir=DEMO / 'roi',
    labels_csv=DEMO / 'labels.csv',
    out_dir=DEMO / 'runs',
    n_folds=3, epochs=15, lr=1e-4, batch_size=4, patience=6,
    dropout=0.2, device='cuda', pretrained=None,
)

## 2B. GERÇEK MOD — kendi verinle eğit

ROI'lerini Drive'a koy: `MyDrive/pf-tumor/data/roi/*.npy` + `MyDrive/pf-tumor/labels.csv`
(labels: `case_id,patient_id,label` — MB/PA/EP). Kod GitHub'dan, **veri** Drive'dan gelir.
`pretrained=` ile CBTN/başka ön-eğitim ağırlığından transfer learning yapılabilir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DATA = Path('/content/drive/MyDrive/pf-tumor')
ROI_DIR = DATA / 'data' / 'roi'
LABELS  = DATA / 'labels.csv'
assert ROI_DIR.is_dir() and LABELS.is_file(), 'data/roi veya labels.csv Drive icinde bulunamadi.'

summary = run(
    roi_dir=ROI_DIR,
    labels_csv=LABELS,
    out_dir=DATA / 'runs' / 'exp1',
    n_folds=5, epochs=150, lr=1e-4, batch_size=4, patience=20,
    dropout=0.2, device='cuda', pretrained=None,
)

## 3. Sonuçlar (demo ya da gerçek — hangi `summary` üretildiyse)

In [ ]:
import numpy as np
print(f"Hasta: {summary['n_patients']} | Örnek: {summary['n_samples']} | Dağılım: {summary['class_distribution']}")
print(f"CV macro-F1: {summary['cv_macro_f1_mean']:.3f} ± {summary['cv_macro_f1_std']:.3f}\n")
for f in summary['folds']:
    print(f"  fold {f['fold']}: best_epoch={f['best_epoch']}  macro_f1={f['best_macro_f1']:.3f}")

from src.classification.model import CLASSES
cm = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
for f in summary['folds']:
    best = max(f['history'], key=lambda h: h['macro_f1'])
    cm += np.array(best['confusion'])
print('\nSınıflar:', CLASSES)
print('Confusion matrix (satır=gerçek, sütun=tahmin):')
print(cm)

## Sonraki adım

Checkpoint'ler `runs/.../foldN_best.pth`. İnference (tek hasta tahmini) hafif — yerel
makinede/klinik bilgisayarda GPU'suz çalışır. Demo'da model sinyali öğreniyorsa
(macro-F1 yükseliyorsa) altyapı doğru; gerçek performans gerçek veriyle ölçülür.